# Hello, Embeddings — turning words into vectors, and measuring meaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/production-ai-engineering/blob/main/notebooks/hello_embeddings.ipynb)

Pairs with **ML_Study_14 / 13**. An **embedding** turns a word (or sentence) into a list of numbers (a vector)
so that **similar meanings sit close together**. We measure "close" with **cosine similarity**. Demo goal:
show that *king* and *queen* are close, while *king* and *laptop* are far apart.

*First run downloads a small real word-vector model (~66MB, GloVe). Works in Colab as-is.*

## 1. Cosine similarity — the mechanic (works on ANY vectors)

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))   # 1 = same dir, 0 = unrelated

# toy 3-D vectors to see the idea before real embeddings
print("same direction :", cosine([1,0,0], [2,0,0]))     # 1.0
print("orthogonal     :", cosine([1,0,0], [0,1,0]))     # 0.0
print("opposite       :", cosine([1,0,0], [-1,0,0]))    # -1.0
print("cosine DISTANCE = 1 - cosine similarity  (0 = identical meaning)")

## 2. Load a real embedding model (GloVe word vectors)
*Colab has `gensim` preinstalled. If the import fails, run once in a cell:* `!pip install gensim`

In [ ]:
import gensim.downloader as api
wv = api.load("glove-wiki-gigaword-50")   # 400k words, each a 50-number vector (~66MB first run)
print("vocabulary:", len(wv), "words   |   vector size:", wv["king"].shape)

## 3. The 'embedded value' of a word

In [ ]:
vec = wv["king"]
print("The word 'king' as 50 numbers:\n")
print(np.round(vec, 3))
print("\nThat vector IS the model's understanding of 'king'. Meaning lives in the numbers.")

## 4. The demo — king & queen are close; king & laptop are far

In [ ]:
pairs = [("king","queen"), ("king","laptop"), ("cat","dog"), ("cat","laptop"),
         ("paris","france"), ("doctor","nurse"), ("king","man")]
print(f"{'word A':>8}  {'word B':>8}   cosine_sim   cosine_dist")
print("-"*46)
for a, b in pairs:
    s = wv.similarity(a, b)
    tag = "  <- CLOSE" if s > 0.6 else ("  <- far" if s < 0.35 else "")
    print(f"{a:>8}  {b:>8}      {s:5.3f}        {1-s:5.3f}{tag}")

### Rank a word against several others

In [ ]:
def closest(word, others):
    ranked = sorted(others, key=lambda w: -wv.similarity(word, w))
    print(f"closest to '{word}':")
    for w in ranked:
        print(f"   {w:<10} cos={wv.similarity(word, w):.3f}")

closest("king", ["queen", "prince", "man", "castle", "laptop", "banana"])

## 5. The famous analogy: king − man + woman ≈ ?

In [ ]:
result = wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print("king - man + woman  ≈ ", [f"{w} ({s:.2f})" for w, s in result])
print("\nThe vectors encode RELATIONSHIPS, not just words — 'royalty' and 'gender' are directions in the space.")

## 6. See it — words that mean similar things cluster together (2-D)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
%matplotlib inline

groups = {
    "royalty": ["king","queen","prince","throne","crown"],
    "tech":    ["laptop","computer","keyboard","software","internet"],
    "animals": ["cat","dog","horse","cow","elephant"],
}
words = [w for g in groups.values() for w in g]
X = np.array([wv[w] for w in words])
xy = PCA(n_components=2, random_state=0).fit_transform(X)

colors = {"royalty":"#d62728","tech":"#1f77b4","animals":"#2ca02c"}
plt.figure(figsize=(8,5.5))
k = 0
for g, ws in groups.items():
    pts = xy[k:k+len(ws)]; k += len(ws)
    plt.scatter(pts[:,0], pts[:,1], c=colors[g], label=g, s=60)
    for (x,y), w in zip(pts, ws):
        plt.annotate(w, (x,y), textcoords="offset points", xytext=(5,4), fontsize=9)
plt.title("Meaning is geometry: similar words land near each other")
plt.legend(); plt.tight_layout(); plt.show()

## Takeaway
- An **embedding** maps text → a vector; **similar meaning → nearby vectors**.
- **Cosine similarity** (1 = same, 0 = unrelated) measures that closeness; **cosine distance = 1 − similarity**.
- *king/queen* ≈ 0.78 (close), *king/laptop* ≈ 0.20 (far) — the model learned meaning from data.
- This is the engine under **semantic search and RAG** (Study 13): embed the question, find the nearest chunks.
- Modern apps use richer **sentence** embeddings (e.g. `sentence-transformers`, or an embeddings API) — same idea, bigger vectors.